# Лабораторная работа 5. Деревья решений

Цель: реализовать дерево решений для классификации (критерий Джини, прирост информации, критерии останова, метрика точности) и сравнить с `DecisionTreeClassifier` из sklearn. Вариант 17.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score

RANDOM_STATE = 17
MAX_DEPTH = 5
MIN_SAMPLES_LEAF = 3
MIN_SAMPLES_SPLIT = 6

## 1. Данные

Синтетическая бинарная классификация `make_classification`: 400 объектов, 4 признака, `random_state=17`. Разбиение 70/30 со стратификацией.

In [ ]:
X, y = make_classification(
    n_samples=400,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Train classes:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Test classes:', dict(zip(*np.unique(y_test, return_counts=True))))

## 2. Узел и лист

In [ ]:
class Node:
    def __init__(self, index, t, true_branch, false_branch):
        self.index = index  # индекс признака
        self.t = t  # порог
        self.true_branch = true_branch
        self.false_branch = false_branch


class Leaf:
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
        self.prediction = self.predict()

    def predict(self):
        classes = {}
        for label in self.labels:
            if label not in classes:
                classes[label] = 0
            classes[label] += 1
        return max(classes, key=classes.get)

## 3. Критерий Джини и прирост информации

In [ ]:
def gini(labels):
    """Критерий Джини: 1 - sum(p_k^2)."""
    labels = np.asarray(labels)
    n = labels.shape[0]
    if n == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / n
    return 1.0 - float(np.sum(probabilities ** 2))


def gain(left_labels, right_labels, root_gini):
    """Прирост информации (уменьшение неопределённости Джини)."""
    n = len(left_labels) + len(right_labels)
    if n == 0:
        return 0.0
    p_left = len(left_labels) / n
    p_right = len(right_labels) / n
    return root_gini - p_left * gini(left_labels) - p_right * gini(right_labels)


print('gini([0,0,1,1]) =', round(gini([0, 0, 1, 1]), 4))
print('gini([0,0,0,0]) =', round(gini([0, 0, 0, 0]), 4))

## 4. Разбиение и поиск лучшего сплита

Пороги — середины между соседними уникальными значениями признака (как в sklearn). В разбиении не допускаются дочерние узлы меньше `min_samples_leaf`.

In [ ]:
def split(data, labels, column_index, t):
    left = np.where(data[:, column_index] <= t)
    right = np.where(data[:, column_index] > t)
    true_data = data[left]
    false_data = data[right]
    true_labels = labels[left]
    false_labels = labels[right]
    return true_data, false_data, true_labels, false_labels


def find_best_split(data, labels, min_samples_leaf=MIN_SAMPLES_LEAF):
    root_gini = gini(labels)
    best_gain = 0.0
    best_t = None
    best_index = None
    n_features = data.shape[1]

    for index in range(n_features):
        values = np.unique(data[:, index])
        if values.shape[0] < 2:
            continue
        t_values = (values[:-1] + values[1:]) / 2.0

        for t in t_values:
            true_data, false_data, true_labels, false_labels = split(
                data, labels, index, t
            )
            if (
                true_data.shape[0] < min_samples_leaf
                or false_data.shape[0] < min_samples_leaf
            ):
                continue

            current_gain = gain(true_labels, false_labels, root_gini)
            # При равном приросте берём последний кандидат (как в sklearn)
            if current_gain >= best_gain and current_gain > 0:
                best_gain, best_t, best_index = current_gain, t, index

    return best_gain, best_t, best_index

## 5. Построение дерева и критерии останова

Останов: (1) достигнута `max_depth`; (2) в узле меньше `min_samples_split` объектов; (3) прирост информации равен нулю (нет допустимого сплита).

In [ ]:
def build_tree(
    data,
    labels,
    depth=0,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
):
    if depth >= max_depth or data.shape[0] < min_samples_split:
        return Leaf(data, labels)

    split_gain, t, index = find_best_split(
        data, labels, min_samples_leaf=min_samples_leaf
    )

    if split_gain == 0 or t is None or index is None:
        return Leaf(data, labels)

    true_data, false_data, true_labels, false_labels = split(data, labels, index, t)
    true_branch = build_tree(
        true_data, true_labels,
        depth=depth + 1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
    )
    false_branch = build_tree(
        false_data, false_labels,
        depth=depth + 1,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
    )
    return Node(index, t, true_branch, false_branch)


def classify_object(obj, node):
    if isinstance(node, Leaf):
        return node.prediction
    if obj[node.index] <= node.t:
        return classify_object(obj, node.true_branch)
    return classify_object(obj, node.false_branch)


def predict(data, tree):
    return [classify_object(obj, tree) for obj in data]


def print_tree(node, spacing=''):
    if isinstance(node, Leaf):
        print(spacing + 'Прогноз:', node.prediction)
        return
    print(spacing + 'Индекс', str(node.index), '<=', str(node.t))
    print(spacing + '--> True:')
    print_tree(node.true_branch, spacing + '  ')
    print(spacing + '--> False:')
    print_tree(node.false_branch, spacing + '  ')


def tree_depth(node):
    if isinstance(node, Leaf):
        return 0
    return 1 + max(tree_depth(node.true_branch), tree_depth(node.false_branch))


def count_leaves(node):
    if isinstance(node, Leaf):
        return 1
    return count_leaves(node.true_branch) + count_leaves(node.false_branch)

## 6. Метрика точности

In [ ]:
def accuracy_metric(actual, predicted):
    """Доля правильных ответов."""
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    return float(np.sum(actual == predicted) / actual.shape[0])

## 7. Обучение самописного дерева

In [ ]:
my_tree = build_tree(
    X_train,
    y_train,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
)

y_train_pred = predict(X_train, my_tree)
y_test_pred = predict(X_test, my_tree)
acc_train = accuracy_metric(y_train, y_train_pred)
acc_test = accuracy_metric(y_test, y_test_pred)

print(f'Глубина: {tree_depth(my_tree)}, листьев: {count_leaves(my_tree)}')
print(f'Accuracy train: {acc_train:.6f}')
print(f'Accuracy test:  {acc_test:.6f}')
print('Структура дерева:')
print_tree(my_tree)

## 8. Сравнение с sklearn

Те же гиперпараметры: `criterion='gini'`, `max_depth=5`, `min_samples_leaf=3`, `min_samples_split=6`.

In [ ]:
sk_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    min_samples_split=MIN_SAMPLES_SPLIT,
    random_state=RANDOM_STATE,
)
sk_tree.fit(X_train, y_train)

sk_train_pred = sk_tree.predict(X_train)
sk_test_pred = sk_tree.predict(X_test)
sk_acc_train = accuracy_score(y_train, sk_train_pred)
sk_acc_test = accuracy_score(y_test, sk_test_pred)

print(f'Глубина: {sk_tree.get_depth()}, листьев: {sk_tree.get_n_leaves()}')
print(f'Accuracy train: {sk_acc_train:.6f}')
print(f'Accuracy test:  {sk_acc_test:.6f}')
print()
print(export_text(sk_tree, feature_names=[f'x{i}' for i in range(X.shape[1])]))

train_match = np.array_equal(y_train_pred, sk_train_pred)
test_match = np.array_equal(y_test_pred, sk_test_pred)
print('Совпадение прогнозов train:', train_match)
print('Совпадение прогнозов test: ', test_match)
print('|Δ accuracy train|:', abs(acc_train - sk_acc_train))
print('|Δ accuracy test|: ', abs(acc_test - sk_acc_test))

assert train_match and test_match, 'Прогнозы должны совпасть полностью'
print('OK: результаты совпали полностью.')